# 19 — Frozen all-six-variable official-test evaluation

Evaluate the selected all-six-variable baseline v1 checkpoint once,
using its validation-selected threshold without modification.
Report overall and per-year official-test metrics.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.7"
VALIDATION_FRACTION = 0.20
VALIDATION_SEED = 20260913
YEARS = tuple(range(2013, 2023))
VARIABLES = (
    "DBZ",
    "KDP",
    "RHOHV",
    "VEL",
    "WIDTH",
    "ZDR",
)
CHANNEL_ORDER = tuple(
    f"{variable}_sweep_{sweep}"
    for variable in VARIABLES
    for sweep in range(2)
)
CHANNEL_COUNT = len(CHANNEL_ORDER)

FILE_BATCH_SIZE = 16
NUM_WORKERS = 8

FROZEN_CHECKPOINT_EPOCH = 20
FROZEN_THRESHOLD = (
    0.8699302077293396
)

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / (
        f"tornet_detection-"
        f"{PACKAGE_VERSION}-py3-none-any.whl"
    )
)
MANIFESTS_ROOT = (
    BACKUP_ROOT / "manifests"
)
NORMALIZATION_PATH = (
    BACKUP_ROOT
    / "experiments"
    / "all_year_all6_baseline_v1"
    / "normalization.json"
)
EXPERIMENT_DIRECTORY = (
    BACKUP_ROOT
    / "experiments"
    / "all_year_all6_baseline_v1"
)
CHECKPOINT_PATH = (
    EXPERIMENT_DIRECTORY
    / "best_model.pt"
)
VALIDATION_METRICS_PATH = (
    EXPERIMENT_DIRECTORY
    / "validation_metrics.json"
)
TEST_METRICS_PATH = (
    EXPERIMENT_DIRECTORY
    / "official_test_metrics.json"
)
EXTRACTION_ROOT = Path(
    "/content/"
    "tornet_all6_official_test"
)

for path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    NORMALIZATION_PATH,
    CHECKPOINT_PATH,
    VALIDATION_METRICS_PATH,
):
    if not path.exists():
        raise FileNotFoundError(path)

if TEST_METRICS_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite "
        f"{TEST_METRICS_PATH}"
    )

print("checkpoint:", CHECKPOINT_PATH)
print(
    "frozen epoch:",
    FROZEN_CHECKPOINT_EPOCH,
)
print(
    "frozen threshold:",
    FROZEN_THRESHOLD,
)

checkpoint: /content/drive/MyDrive/TorNet_Backup/experiments/all_year_all6_baseline_v1/best_model.pt
frozen epoch: 20
frozen threshold: 0.8699302077293396


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
        "scikit-learn>=1.5",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.7-py3-none-any.whl'], returncode=0)

In [4]:
import datetime
import json
import shutil
import tarfile
import time

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import (
    DataLoader,
    Dataset,
)

import tornado_detection
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
    read_netcdf_file,
)

if (
    tornado_detection.__version__
    != PACKAGE_VERSION
):
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "Select an A100 GPU runtime "
        "and run all cells"
    )

device = torch.device("cuda")

normalization = json.loads(
    NORMALIZATION_PATH.read_text()
)
validation_metrics = json.loads(
    VALIDATION_METRICS_PATH.read_text()
)

if (
    validation_metrics.get(
        "checkpoint_epoch"
    )
    != FROZEN_CHECKPOINT_EPOCH
):
    raise AssertionError(
        "Checkpoint epoch provenance "
        "mismatch"
    )

if (
    validation_metrics.get(
        "selected_threshold"
    )
    != FROZEN_THRESHOLD
):
    raise AssertionError(
        "Threshold provenance mismatch"
    )

if (
    validation_metrics.get(
        "official_test_evaluated"
    )
    is not False
):
    raise AssertionError(
        "Validation artifact already "
        "marks test evaluated"
    )

expected_variables = list(VARIABLES)
expected_channel_order = list(
    CHANNEL_ORDER
)

if (
    validation_metrics.get("variables")
    != expected_variables
):
    raise AssertionError(
        "Validation variable provenance "
        "mismatch"
    )

if (
    validation_metrics.get(
        "channel_order"
    )
    != expected_channel_order
):
    raise AssertionError(
        "Validation channel provenance "
        "mismatch"
    )

if (
    normalization.get("variables")
    != expected_variables
    or normalization.get(
        "channel_order"
    )
    != expected_channel_order
    or normalization.get("tensor_shape")
    != [120, 240, CHANNEL_COUNT]
):
    raise AssertionError(
        "Normalization provenance mismatch"
    )

means = np.asarray(
    normalization["means"],
    dtype=np.float32,
)
stds = np.asarray(
    normalization[
        "standard_deviations"
    ],
    dtype=np.float32,
)

assert means.shape == (CHANNEL_COUNT,)
assert stds.shape == (CHANNEL_COUNT,)

canonical = (
    load_canonical_frame_index(
        MANIFESTS_ROOT
    )
)
assigned = assign_model_splits(
    canonical,
    validation_fraction=(
        VALIDATION_FRACTION
    ),
    seed=VALIDATION_SEED,
)

test_rows = (
    assigned.loc[
        assigned["model_split"].eq(
            "test"
        )
    ]
    .sort_values(
        [
            "archive_member",
            "frame_index",
        ]
    )
    .reset_index(drop=True)
)

assert len(test_rows) == 125_868
assert int(
    test_rows["frame_label"].sum()
) == 3_909

if not (
    test_rows.groupby(
        "archive_member"
    ).size().eq(4).all()
):
    raise AssertionError(
        "Every test file must "
        "contain four frames"
    )

test_records = [
    (
        member,
        group["frame_label"]
        .astype(np.uint8)
        .to_numpy(),
    )
    for member, group
    in test_rows.groupby(
        "archive_member",
        sort=True,
    )
]

required_by_year = {
    year: set(
        test_rows.loc[
            test_rows["year"].eq(year),
            "archive_member",
        ].unique()
    )
    for year in YEARS
}

print(
    "GPU:",
    torch.cuda.get_device_name(0),
)
print(
    "official-test files:",
    len(test_records),
)
print(
    "official-test frames:",
    len(test_rows),
)
print(
    "official-test positives:",
    int(
        test_rows[
            "frame_label"
        ].sum()
    ),
)

GPU: NVIDIA A100-SXM4-40GB
official-test files: 31467
official-test frames: 125868
official-test positives: 3909


In [5]:
if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

staging = []

for year in YEARS:
    drive_archive = (
        BACKUP_ROOT
        / f"tornet_{year}.tar.gz"
    )
    local_archive = Path(
        f"/content/tornet_{year}.tar.gz"
    )

    if not drive_archive.is_file():
        raise FileNotFoundError(
            drive_archive
        )

    if local_archive.exists():
        local_archive.unlink()

    copy_started = (
        time.perf_counter()
    )

    shutil.copyfile(
        drive_archive,
        local_archive,
    )

    copy_seconds = (
        time.perf_counter()
        - copy_started
    )

    required = required_by_year[year]
    extracted = set()
    extraction_started = (
        time.perf_counter()
    )

    with tarfile.open(
        local_archive,
        mode="r:gz",
    ) as archive:
        for member in archive:
            if (
                not member.isfile()
                or member.name
                not in required
            ):
                continue

            destination = (
                EXTRACTION_ROOT
                / member.name
            )
            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            source_file = (
                archive.extractfile(
                    member
                )
            )

            if source_file is None:
                raise RuntimeError(
                    member.name
                )

            with (
                source_file,
                destination.open("wb")
                as output_file,
            ):
                shutil.copyfileobj(
                    source_file,
                    output_file,
                    length=1024 * 1024,
                )

            extracted.add(
                member.name
            )

    extraction_seconds = (
        time.perf_counter()
        - extraction_started
    )
    missing = required - extracted

    if missing:
        raise RuntimeError(
            f"Year {year} missing: "
            f"{sorted(missing)[:10]}"
        )

    local_archive.unlink()

    staging.append(
        {
            "year": year,
            "file_count": (
                len(extracted)
            ),
            "copy_seconds": (
                copy_seconds
            ),
            "extraction_seconds": (
                extraction_seconds
            ),
        }
    )

    print(
        f"year={year} "
        f"staged_files="
        f"{len(extracted):,} "
        f"copy={copy_seconds:.1f}s "
        f"extract="
        f"{extraction_seconds:.1f}s"
    )

print(
    "total staged files:",
    sum(
        row["file_count"]
        for row in staging
    ),
)

staged_bytes = sum(
    path.stat().st_size
    for path
    in EXTRACTION_ROOT.rglob("*.nc")
)

print(
    "staged GiB:",
    round(
        staged_bytes / 1024**3,
        3,
    ),
)

year=2013 staged_files=573 copy=34.4s extract=9.5s
year=2014 staged_files=2,546 copy=166.9s extract=44.2s
year=2015 staged_files=3,902 copy=323.9s extract=52.0s
year=2016 staged_files=2,951 copy=182.8s extract=48.5s
year=2017 staged_files=3,145 copy=273.3s extract=44.5s
year=2018 staged_files=2,518 copy=161.4s extract=38.2s
year=2019 staged_files=4,031 copy=190.1s extract=57.0s
year=2020 staged_files=4,756 copy=289.7s extract=50.9s
year=2021 staged_files=4,268 copy=249.1s extract=55.4s
year=2022 staged_files=2,777 copy=247.4s extract=59.5s
total staged files: 31467
staged GiB: 23.93


In [6]:
class FileDataset(Dataset):
    def __init__(
        self,
        records,
        root,
        channel_means,
        channel_stds,
    ):
        self.records = records
        self.root = root
        self.means = (
            channel_means.reshape(
                1,
                1,
                1,
                CHANNEL_COUNT,
            )
        )
        self.stds = (
            channel_stds.reshape(
                1,
                1,
                1,
                CHANNEL_COUNT,
            )
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        (
            member,
            expected_labels,
        ) = self.records[index]

        result = read_netcdf_file(
            self.root / member,
            variables=VARIABLES,
        )

        np.testing.assert_array_equal(
            result.labels,
            expected_labels,
        )

        values = np.nan_to_num(
            (
                result.values
                - self.means
            )
            / self.stds,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(
            np.float32,
            copy=False,
        )

        inputs = (
            torch.from_numpy(values)
            .permute(0, 3, 1, 2)
            .contiguous()
        )
        labels = torch.from_numpy(
            result.labels.astype(
                np.float32,
                copy=False,
            )
        ).reshape(4, 1)

        return inputs, labels


class RadarBaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                CHANNEL_COUNT,
                16,
                3,
                padding=1,
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                16,
                32,
                3,
                padding=1,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                32,
                64,
                3,
                padding=1,
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                64,
                128,
                3,
                padding=1,
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

        self.classifier = (
            nn.Sequential(
                nn.Flatten(),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(64, 1),
            )
        )

    def forward(self, inputs):
        return self.classifier(
            self.features(inputs)
        )


test_dataset = FileDataset(
    test_records,
    EXTRACTION_ROOT,
    means,
    stds,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=FILE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

model = RadarBaselineCNN().to(
    device
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

if (
    int(checkpoint["epoch"])
    != FROZEN_CHECKPOINT_EPOCH
):
    raise AssertionError(
        "Loaded checkpoint epoch "
        "mismatch"
    )

model.load_state_dict(
    checkpoint["model_state_dict"]
)
model.eval()

print(
    "parameters:",
    f"{sum(parameter.numel() for parameter in model.parameters()):,}",
)

parameters: 107,537


In [7]:
labels = []
probabilities = []

evaluation_started = (
    time.perf_counter()
)

with torch.no_grad():
    for inputs, targets in test_loader:
        files, frames = (
            inputs.shape[:2]
        )

        inputs = inputs.reshape(
            files * frames,
            CHANNEL_COUNT,
            120,
            240,
        ).to(
            device,
            non_blocking=True,
        )

        logits = model(inputs)

        labels.extend(
            targets.numpy()
            .reshape(-1)
            .tolist()
        )
        probabilities.extend(
            logits.sigmoid()
            .cpu()
            .numpy()
            .reshape(-1)
            .tolist()
        )

evaluation_seconds = (
    time.perf_counter()
    - evaluation_started
)

labels = np.asarray(
    labels,
    dtype=np.int64,
)
probabilities = np.asarray(
    probabilities,
    dtype=np.float64,
)

np.testing.assert_array_equal(
    labels,
    test_rows["frame_label"]
    .astype(np.int64)
    .to_numpy(),
)


def calculate_metrics(
    metric_labels,
    metric_probabilities,
):
    predictions = (
        metric_probabilities
        >= FROZEN_THRESHOLD
    ).astype(np.int64)

    precision = float(
        precision_score(
            metric_labels,
            predictions,
            zero_division=0,
        )
    )
    recall = float(
        recall_score(
            metric_labels,
            predictions,
            zero_division=0,
        )
    )
    f1 = float(
        2
        * precision
        * recall
        / max(
            precision + recall,
            1e-12,
        )
    )

    matrix = confusion_matrix(
        metric_labels,
        predictions,
        labels=[0, 1],
    )
    (
        true_negative,
        false_positive,
        false_negative,
        true_positive,
    ) = [
        int(value)
        for value in matrix.ravel()
    ]

    return {
        "frame_count": int(
            len(metric_labels)
        ),
        "positive_count": int(
            metric_labels.sum()
        ),
        "pr_auc": float(
            average_precision_score(
                metric_labels,
                metric_probabilities,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                metric_labels,
                metric_probabilities,
            )
        ),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_negative": (
            true_negative
        ),
        "false_positive": (
            false_positive
        ),
        "false_negative": (
            false_negative
        ),
        "true_positive": (
            true_positive
        ),
    }


overall = calculate_metrics(
    labels,
    probabilities,
)

by_year = []

for year in YEARS:
    mask = (
        test_rows["year"].to_numpy()
        == year
    )

    year_metrics = {
        "year": year
    }
    year_metrics.update(
        calculate_metrics(
            labels[mask],
            probabilities[mask],
        )
    )
    by_year.append(
        year_metrics
    )

print(
    "official-test overall:",
    json.dumps(
        overall,
        indent=2,
        sort_keys=True,
    ),
)
print(
    "evaluation seconds:",
    round(
        evaluation_seconds,
        3,
    ),
)

official-test overall: {
  "f1": 0.418567335243553,
  "false_negative": 2083,
  "false_positive": 2990,
  "frame_count": 125868,
  "positive_count": 3909,
  "pr_auc": 0.39842204114065594,
  "precision": 0.3791528239202658,
  "recall": 0.46712714249168585,
  "roc_auc": 0.8896639555051287,
  "true_negative": 118969,
  "true_positive": 1826
}
evaluation seconds: 435.971


In [8]:
artifact = {
    "artifact_kind": (
        "all_year_all6_baseline_v1_"
        "official_test_metrics"
    ),
    "created_at_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "package_version": (
        PACKAGE_VERSION
    ),
    "variables": list(VARIABLES),
    "channel_order": list(
        CHANNEL_ORDER
    ),
    "channel_count": CHANNEL_COUNT,
    "selected_model": (
        "all_year_all6_baseline_v1"
    ),
    "checkpoint_epoch": (
        FROZEN_CHECKPOINT_EPOCH
    ),
    "threshold_source": (
        "all_year_all6_baseline_v1 "
        "validation"
    ),
    "frozen_threshold": (
        FROZEN_THRESHOLD
    ),
    "official_test_evaluated": True,
    "evaluation_seconds": (
        evaluation_seconds
    ),
    "overall": overall,
    "by_year": by_year,
    "staging": staging,
}

TEST_METRICS_PATH.write_text(
    json.dumps(
        artifact,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

print(
    json.dumps(
        artifact,
        indent=2,
        sort_keys=True,
    )
)
print(
    "wrote:",
    TEST_METRICS_PATH,
)

{
  "artifact_kind": "all_year_all6_baseline_v1_official_test_metrics",
  "by_year": [
    {
      "f1": 0.7262247838616714,
      "false_negative": 31,
      "false_positive": 64,
      "frame_count": 2292,
      "positive_count": 157,
      "pr_auc": 0.7416336940108356,
      "precision": 0.6631578947368421,
      "recall": 0.802547770700637,
      "roc_auc": 0.9711943793911006,
      "true_negative": 2071,
      "true_positive": 126,
      "year": 2013
    },
    {
      "f1": 0.5693606755126658,
      "false_negative": 320,
      "false_positive": 394,
      "frame_count": 10184,
      "positive_count": 792,
      "pr_auc": 0.5773111116353796,
      "precision": 0.5450346420323325,
      "recall": 0.5959595959595959,
      "roc_auc": 0.90123323847504,
      "true_negative": 8998,
      "true_positive": 472,
      "year": 2014
    },
    {
      "f1": 0.4805013927576602,
      "false_negative": 325,
      "false_positive": 421,
      "frame_count": 15608,
      "positive_count": 670

In [9]:
shutil.rmtree(EXTRACTION_ROOT)

assert not EXTRACTION_ROOT.exists()
assert TEST_METRICS_PATH.is_file()

print(
    "Removed all Colab-local "
    "official-test data"
)
print(
    "Preserved:",
    TEST_METRICS_PATH,
)

Removed all Colab-local official-test data
Preserved: /content/drive/MyDrive/TorNet_Backup/experiments/all_year_all6_baseline_v1/official_test_metrics.json
